Requirement: n1-highmem-2

# Getting started

In [1]:
# Use the os package to interact with the environment
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import glob
import sys as sys

In [2]:
# show all columns when printing
pd.set_option('display.max_columns', None) 

In [3]:
#set releases
RAW_GENO = '/home/jupyter/workspace/path/to/release7/raw_genotypes'
CLINICAL = '/home/jupyter/workspace/path/to/release7/clinical_data'
RELATED = '/home/jupyter/workspace/path/to/release7/meta_data/related_samples'

# Install Packages

In [6]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_x86_64_20250122.zip

unzip -o plink2_linux_x86_64_20250122.zip

fi

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

In [ ]:
%%capture
%%bash

#download hap-ibd
wget https://faculty.washington.edu/browning/hap-ibd.jar

#download beagle 5.4
wget https://faculty.washington.edu/browning/beagle/beagle.29Oct24.c8e.jar

#download human genetic map required by beagle
wget https://bochet.gcc.biostat.washington.edu/beagle/genetic_maps/plink.GRCh38.map.zip

unzip plink.GRCh38.map.zip


# Phase chr12 in European population

In [ ]:
%%bash

#convert plink pfile to vcf for phasing

RAW_GENO='/home/jupyter/workspace/path/to/release7/raw_genotypes'

/home/jupyter/plink2 --pfile ${RAW_GENO}/EUR/EUR_release7_vwb \
                     --chr 12 \
                     --geno 0.01 \
                     --mind 0.01 \
                     --from-bp 40000000 \
                     --to-bp 43000000 \
                     --hwe 1E-5 1000 \
                     --set-all-var-ids @:#:$r:$a \
                     --rm-dup exclude-mismatch \
                     --snps-only just-acgt \
                     --export vcf id-paste=iid \
                     --out eur_chr12


In [6]:
%%bash 

bgzip -c eur_chr12.vcf > eur_chr12.vcf.gz
tabix -p vcf -f eur_chr12.vcf.gz

In [7]:
%%bash

java -jar /home/jupyter/beagle.29Oct24.c8e.jar \
	gt=eur_chr12.vcf.gz \
	map=plink.chr12.GRCh38.map \
	out=eur_chr12_phased

beagle.29Oct24.c8e.jar (version 5.4)
Copyright (C) 2014-2022 Brian L. Browning
Enter "java -jar beagle.29Oct24.c8e.jar" to list command line argument
Start time: 03:16 PM GMT on 16 May 2025

Command line: java -Xmx3244m -jar beagle.29Oct24.c8e.jar
  gt=eur_chr12.vcf.gz
  map=plink.chr12.GRCh38.map
  out=eur_chr12_phased.vcf.gz
  nthreads=2

Reference samples:                    0
Study     samples:               37,999

Window 1 [12:40004913-42996867]
Study     markers:                1,747

Burnin  iteration 1:           1 minute 44 seconds
Burnin  iteration 2:           1 minute 37 seconds

Estimated ne:                  2913603
Estimated err:                 7.4e-05

Phasing iteration 1:           1 minute 34 seconds
Phasing iteration 2:           1 minute 32 seconds
Phasing iteration 3:           1 minute 29 seconds
Phasing iteration 4:           1 minute 27 seconds
Phasing iteration 5:           1 minute 26 seconds
Phasing iteration 6:           1 minute 27 seconds
Phasing iterati

In [9]:
%%bash

java -jar /home/jupyter/hap-ibd.jar \
        gt=eur_chr12_phased.vcf.gz \
        map=plink.chr12.GRCh38.map \
        out=eur_chr12_ibd

Copyright (C) 2019-2023 Brian L. Browning
Enter "java -jar hap-ibd.jar" to print a list of command line arguments

Program            :  hap-ibd.jar  [ version 1.0, 15Jun23.92f ]
Start Time         :  03:43 PM GMT on 16 May 2025
Max Memory         :  3244 MB

Parameters
  gt               :  eur_chr12_phased.vcf.gz
  map              :  plink.chr12.GRCh38.map
  out              :  eur_chr12_ibd
  min-seed         :  2.0
  max-gap          :  1000
  min-extend       :  1.0
  min-output       :  2.0
  min-markers      :  100
  min-mac          :  2
  nthreads         :  2

Statistics
  samples          :  37999
  markers          :  1410
  IBD segments     :  114509
  IBD segs/sample  :  3.0
  HBD segments     :  38
  HBD segs/sample  :  0.001

Wallclock Time:    :  14 seconds
End Time           :  03:43 PM GMT on 16 May 2025


# Find ibd across the variant carriers

In [13]:
# get carrier list
carrier_list = pd.read_csv('LRRK2_L1795F_carriers.csv')

# read ibd results
ibd=pd.read_csv('eur_chr12_ibd.ibd.gz',compression='gzip',sep='\t', header=None,names=['sample1','hap1','sample2','hap2','chrom','Start','End','cM_len'])

# subset just ibd segement among the carriers
keep_ibd=ibd.loc[(ibd['sample1'].isin(carrier_list['sample'])) & (ibd['sample2'].isin(carrier_list['sample']))]

keep_ibd.to_csv('ibd_sharing.csv',index=False)


# generate IBD plot

In [ ]:
%%R

library(ggplot2)
library(dplyr)
library(RColorBrewer)


data=read.csv('ibd_sharing.csv')

data <- data %>%
  arrange(FAM) %>%  # First sort by val. This sort the dataframe but NOT the factor levels
  mutate(SegmentID = row_number())%>%
  mutate(SegmentLength = End - Start) %>%
  arrange(FAM, desc(SegmentLength))

median(data$cM_len)
min(data$cM_len)
max(data$cM_len)

# Plot the IBD segments
ggplot(data, aes(x = Start, xend = End, y = SegmentID, yend = SegmentID, color = FAM)) +
  geom_segment(size = 1) +
  geom_text(aes(x = 1, label = Pair), hjust = 0.5, color = "black", size = 5) +  # Fixed y position for all text
  facet_wrap(~Chrom, scales = "free_x") +
  labs(title = "Overlapping IBD Segments",
       x = "Genomic Position", y = "Segment ID") +
  theme_minimal() +
  theme(panel.border = element_rect(color = "black", fill = NA, linewidth = 1.5),  # Add black border
        axis.title.y=element_blank(),
        axis.text.y = element_blank(),  # Remove Y-axis ticks
        axis.ticks.y = element_blank(),  # Remove Y-axis ticks
        strip.background = element_blank(),  # Remove facet strip background
        strip.text = element_blank(),  # Remove facet strip text
        panel.grid.major = element_blank(), panel.grid.minor = element_blank(),panel.background = element_blank())+
  geom_vline(xintercept = 40322386, color = "dark grey", size = 0.5,linetype="dashed")+
  scale_color_manual(values = c("UR" = "steelblue", "GP2-FAM-1" = "#FC4E07"))+
  labs(title = "title") +
  labs(title = NULL)
